In [13]:
import os
import numpy as np
import pandas as pd

while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print("cwd:", os.getcwd())

files = ['data/processed/openaq_locations_filtered.csv',
         'data/processed/openaq_sensors_selected.csv',
         'data/processed/dead_sensors.csv',
         'data/raw/openaq/openaq_locations.csv',
         'data/raw/openaq/openaq_recent.csv']

for f in files:
    print("\n" + "=" * 70)
    print(f, "| exists:", os.path.exists(f))
    if os.path.exists(f):
        df = pd.read_csv(f, nrows=5000)
        print("columns:", df.columns.tolist())
        print(df.head(3).to_string())

for pkg in ['sklearn', 'folium']:
    try:
        mod = __import__(pkg)
        print("\n", pkg, "OK", getattr(mod, '__version__', ''))
    except ImportError:
        print("\n", pkg, "MISSING")

cwd: C:\Users\91935\PycharmProjects\AQi project

data/processed/openaq_locations_filtered.csv | exists: True
columns: ['location_id', 'name', 'locality', 'latitude', 'longitude', 'sensor_ids', 'parameters', 'is_mobile', 'is_monitor', 'provider', 'city']
   location_id                                          name locality   latitude  longitude                                                                                                                                             sensor_ids                                                                                                                                                            parameters  is_mobile  is_monitor provider   city
0           13  Delhi Technological University, Delhi - CPCB      NaN  28.744000  77.120000                                                                                                                                     [13866, 24, 13864]                                                        

In [14]:
import sys
!{sys.executable} -m pip install folium


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
rec = pd.read_csv('data/raw/openaq/openaq_recent.csv', parse_dates=['date'])
loc = pd.read_csv('data/processed/openaq_locations_filtered.csv')[
    ['location_id', 'latitude', 'longitude', 'is_mobile']]

centres = {'Delhi': (28.61, 77.21), 'Mumbai': (19.08, 72.88), 'Kolkata': (22.57, 88.36),
           'Chennai': (13.08, 80.27), 'Bengaluru': (12.97, 77.59),
           'Lucknow': (26.85, 80.95), 'Patna': (25.59, 85.14)}

def haversine(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = (np.sin((p2 - p1) / 2) ** 2
         + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2)
    return 2 * r * np.arcsin(np.sqrt(a))

st = (rec.groupby(['city', 'location_id', 'location_name'])
         .agg(days=('date', 'nunique'),
              params=('parameter', lambda x: ','.join(sorted(set(x)))))
         .reset_index()
         .merge(loc, on='location_id', how='left'))

st['km_from_centre'] = [
    haversine(r.latitude, r.longitude, *centres[r.city]) if pd.notna(r.latitude) else np.nan
    for r in st.itertuples()]

print("Stations with no coordinates:", int(st['latitude'].isna().sum()))
print("\nStations per city, and how many are over 50 km from the city centre:")
print(st.groupby('city').agg(stations=('location_id', 'nunique'),
                             over_50km=('km_from_centre', lambda x: int((x > 50).sum()))))

print("\nAll stations:")
print(st[['city', 'location_name', 'latitude', 'longitude', 'km_from_centre', 'days', 'params']]
        .sort_values(['city', 'km_from_centre'])
        .round({'km_from_centre': 0, 'latitude': 3, 'longitude': 3})
        .to_string(index=False))

Stations with no coordinates: 0

Stations per city, and how many are over 50 km from the city centre:
           stations  over_50km
city                          
Bengaluru        12          0
Chennai           8          0
Delhi            42          0
Kolkata           7          0
Lucknow           5          0
Mumbai           29          0
Patna             4          2

All stations:
     city                                        location_name  latitude  longitude  km_from_centre  days                  params
Bengaluru                  Hombegowda Nagar, Bengaluru - KSPCB    12.939     77.590             3.0   361                 o3,pm25
Bengaluru               Jayanagar 5th Block, Bengaluru - KSPCB    12.921     77.585             5.0   363                 o3,pm10
Bengaluru                      Bapuji Nagar, Bengaluru - KSPCB    12.952     77.540             6.0   298                      o3
Bengaluru                         BTM Layout, Bengaluru - CPCB    12.914     77.595 

In [16]:
import sys
sys.path.append('src')
from aqi_calculator import aqi_and_dominant

bad_ids = st.loc[(st['city'] == 'Patna') & (st['km_from_centre'] > 50), 'location_id'].tolist()
print("mislabelled location_ids:", bad_ids)

p = rec[rec['city'] == 'Patna'].copy()
p['from_bad_station'] = p['location_id'].isin(bad_ids)
print("\nDays with data per pollutant, by station group (True = mislabelled):")
print(p.groupby(['parameter', 'from_bad_station'])['date'].nunique().unstack(fill_value=0))

pm = {'pm25': 'PM2.5', 'pm10': 'PM10', 'no2': 'NO2', 'so2': 'SO2', 'co': 'CO', 'o3': 'O3'}
bounds = {'PM2.5': (0, 1000), 'PM10': (0, 1000), 'NO2': (0, 500), 'SO2': (0, 500), 'CO': (0, 50), 'O3': (0, 500)}

def daily_aqi(df):
    w = (df.assign(parameter=df['parameter'].map(pm))
           .pivot_table(index='date', columns='parameter', values='value', aggfunc='mean'))
    for c in pm.values():
        if c not in w:
            w[c] = np.nan
        lo, hi = bounds[c]
        w.loc[(w[c] < lo) | (w[c] > hi), c] = np.nan
    w['AQI'], _ = aqi_and_dominant(w, min_pollutants=1)
    w['month'] = w.index.month
    return w

def summarise(w, label):
    s = w['AQI'].dropna()
    winter = w.loc[w['month'].isin([11, 12, 1, 2]), 'AQI'].mean()
    print(f"{label}: days {len(s)}, mean AQI {s.mean():.1f}, winter mean {winter:.1f}, "
          f"days above 300: {(s > 300).sum()}")

all_st = daily_aqi(p)
good_st = daily_aqi(p[~p['from_bad_station']])
summarise(all_st, "All 4 stations      ")
summarise(good_st, "Patna stations only ")

diff = (good_st['AQI'] - all_st['AQI']).dropna()
print(f"\nDays AQI changes: {(diff.abs() > 0.5).sum()} | mean change {diff.mean():.1f} "
      f"| max {diff.abs().max():.1f}")

master = pd.read_csv('data/processed/aqi_weather_master.csv', parse_dates=['date'])
m = master[(master['city'] == 'Patna') & (master['data_source'] == 'openaq_recent')]
print("Patna recent AQI in the master file: mean", round(m['AQI'].mean(), 1),
      "| days with AQI", int(m['AQI'].notna().sum()))

mislabelled location_ids: [5628, 6267807]

Days with data per pollutant, by station group (True = mislabelled):
from_bad_station  False  True 
parameter                     
co                    0     47
no2                   0     47
o3                  365     47
pm10                365    365
pm25                365    365
so2                   0     47
All 4 stations      : days 365, mean AQI 106.9, winter mean 154.2, days above 300: 1
Patna stations only : days 365, mean AQI 81.2, winter mean 119.8, days above 300: 0

Days AQI changes: 356 | mean change -25.6 | max 155.6
Patna recent AQI in the master file: mean 81.2 | days with AQI 365


In [17]:
raw_loc = pd.read_csv('data/raw/openaq/openaq_locations.csv')
raw_loc['km_from_patna'] = haversine(raw_loc['latitude'], raw_loc['longitude'], *centres['Patna'])
near = raw_loc[raw_loc['km_from_patna'] <= 50].copy()

tag = rec.drop_duplicates('location_id').set_index('location_id')['city']
near['in_recent_file'] = near['location_id'].isin(rec['location_id'])
near['tagged_city'] = near['location_id'].map(tag)
print(near[['location_id', 'name', 'provider', 'km_from_patna', 'parameters',
            'in_recent_file', 'tagged_city']]
      .round({'km_from_patna': 1}).to_string(index=False))

 location_id                                       name provider  km_from_patna                                                                                                                                                           parameters  in_recent_file tagged_city
        2593   IGSC Planetarium Complex - Patna - BSPCB     CPCB           26.0                                                                                                                                   ['co', 'no2', 'o3', 'pm25', 'so2']           False         NaN
        5652    IGSC Planetarium Complex, Patna - BSPCB     CPCB            0.5                 ['co', 'co', 'no', 'no2', 'no2', 'nox', 'o3', 'o3', 'pm25', 'pm25', 'relativehumidity', 'so2', 'so2', 'temperature', 'wind_direction', 'wind_speed']           False         NaN
        7495          DRM Office Danapur, Patna - BSPCB     CPCB            9.7 ['co', 'co', 'no', 'no2', 'no2', 'nox', 'o3', 'o3', 'pm10', 'pm10', 'pm25', 'pm25', 'relativehumidity

In [18]:
raw_loc = pd.read_csv('data/raw/openaq/openaq_locations.csv')
filt = pd.read_csv('data/processed/openaq_locations_filtered.csv')
sel = pd.read_csv('data/processed/openaq_sensors_selected.csv')
dead = set(pd.read_csv('data/processed/dead_sensors.csv')['sensor_id'])

live_sel_ids = set(sel.loc[~sel['sensor_id'].isin(dead), 'location_id'])

rows = []
for city, (la, lo) in centres.items():
    d = haversine(raw_loc['latitude'], raw_loc['longitude'], la, lo)
    ids = set(raw_loc.loc[d <= 50, 'location_id'])
    rows.append({'city': city,
                 'raw_within_50km': len(ids),
                 'in_locations_filtered': len(ids & set(filt['location_id'])),
                 'in_sensors_selected': len(ids & set(sel['location_id'])),
                 'has_live_selected_sensor': len(ids & live_sel_ids),
                 'in_recent_file': len(ids & set(rec['location_id']))})
print(pd.DataFrame(rows).to_string(index=False))

patna_ids = [5652, 7495, 10630, 10670, 2593, 10645, 10599, 10697]
print("\nPatna-area stations through the funnel:")
print(pd.DataFrame({
    'location_id': patna_ids,
    'in_filtered': [i in set(filt['location_id']) for i in patna_ids],
    'in_selected': [i in set(sel['location_id']) for i in patna_ids],
    'live_sensor': [i in live_sel_ids for i in patna_ids],
    'in_recent': [i in set(rec['location_id']) for i in patna_ids]}).to_string(index=False))

     city  raw_within_50km  in_locations_filtered  in_sensors_selected  has_live_selected_sensor  in_recent_file
    Delhi              120                     78                   78                        42              42
   Mumbai               54                     40                   40                        29              29
  Kolkata               20                     11                   11                         7               7
  Chennai               15                     11                   11                         8               8
Bengaluru               28                     18                   18                        12              12
  Lucknow               10                      8                    8                         5               5
    Patna               10                      7                    7                         2               2

Patna-area stations through the funnel:
 location_id  in_filtered  in_selected  live_sensor  in

In [19]:
raw_loc = pd.read_csv('data/raw/openaq/openaq_locations.csv')
filt_ids = set(pd.read_csv('data/processed/openaq_locations_filtered.csv')['location_id'])

for city, (la, lo) in centres.items():
    d = haversine(raw_loc['latitude'], raw_loc['longitude'], la, lo)
    missed = raw_loc[(d <= 50) & ~raw_loc['location_id'].isin(filt_ids)].copy()
    missed['km'] = d[missed.index].round(0)
    has_pm = missed['parameters'].str.contains('pm25|pm10', na=False)
    print(f"\n{city}: {len(missed)} stations within 50 km not matched by name, {int(has_pm.sum())} report PM")
    print(missed[has_pm].sort_values('km')[['location_id', 'name', 'provider', 'km']]
          .head(12).to_string(index=False))


Delhi: 42 stations within 50 km not matched by name, 42 report PM
 location_id                              name    provider   km
     4712609                         Air Check AirGradient  3.0
     6145551                         Anand Lok AirGradient  6.0
     2860223              GK1 (Oberoi Terrace) AirGradient  7.0
          16                       Civil Lines        CPCB  8.0
        6980           Sector-1, Noida - UPPCB        CPCB 11.0
     4663956 Santushti Apartments, Vasant Kunj AirGradient 11.0
        5598   Sector - 125, Noida, UP - UPPCB        CPCB 13.0
          15                       IGI Airport        CPCB 13.0
     6299494    Prashant Garden, Khora - UPPCB         NaN 13.0
        6924    Indirapuram, Ghaziabad - UPPCB        CPCB 15.0
        5665     Vasundhara, Ghaziabad - UPPCB        CPCB 15.0
        5616      Sector - 62, Noida, UP - IMD        CPCB 15.0

Mumbai: 14 stations within 50 km not matched by name, 14 report PM
 location_id                     

In [20]:
import sys, importlib
sys.path.append('src')
import aqi_calculator
importlib.reload(aqi_calculator)
from aqi_calculator import aqi_and_dominant

master = pd.read_csv('data/processed/aqi_weather_master.csv', parse_dates=['date'])
master.to_csv('data/processed/aqi_weather_master_v2_backup.csv', index=False)

BAD = [5628, 6267807]
pm_map = {'pm25': 'PM2.5', 'pm10': 'PM10', 'no2': 'NO2', 'so2': 'SO2', 'co': 'CO', 'o3': 'O3'}
bounds = {'PM2.5': (0, 1000), 'PM10': (0, 1000), 'NO2': (0, 500),
          'SO2': (0, 500), 'CO': (0, 50), 'O3': (0, 500)}

p = rec[(rec['city'] == 'Patna') & ~rec['location_id'].isin(BAD)].copy()
w = (p.assign(parameter=p['parameter'].map(pm_map))
       .pivot_table(index='date', columns='parameter', values='value', aggfunc='mean'))
for c, (lo, hi) in bounds.items():
    if c not in w:
        w[c] = np.nan
    w.loc[(w[c] < lo) | (w[c] > hi), c] = np.nan

mask = (master['city'] == 'Patna') & (master['data_source'] == 'openaq_recent')
d = master.loc[mask, 'date']
w = w.reindex(pd.DatetimeIndex(sorted(d.unique()))).ffill(limit=2)
aqi, dom = aqi_and_dominant(w[list(bounds)], min_pollutants=1)

before = master.loc[mask, 'AQI'].copy()
for c in bounds:
    master.loc[mask, c] = d.map(w[c]).values
master.loc[mask, 'calc_aqi'] = d.map(aqi).values
master.loc[mask, 'AQI'] = d.map(aqi).values
master.loc[mask, 'dominant_pollutant'] = d.map(dom).values

after = master.loc[mask, 'AQI']
winter = master.loc[mask & master['month'].isin([11, 12, 1, 2]), 'AQI'].mean()
print("Patna recent rows:", int(mask.sum()))
print("AQI mean before/after:", round(before.mean(), 1), "/", round(after.mean(), 1))
print("Patna recent winter mean after:", round(winter, 1))
print("days with AQI before/after:", int(before.notna().sum()), "/", int(after.notna().sum()))
print("\nPatna recent pollutant coverage (days):")
print(master.loc[mask, list(bounds)].notna().sum())
print("\nDominant pollutant:")
print(master.loc[mask, 'dominant_pollutant'].value_counts(normalize=True).mul(100).round(1))

master.to_csv('data/processed/aqi_weather_master.csv', index=False)
print("\nSaved. Master shape:", master.shape)

Patna recent rows: 365
AQI mean before/after: 81.2 / 81.2
Patna recent winter mean after: 119.8
days with AQI before/after: 365 / 365

Patna recent pollutant coverage (days):
PM2.5    365
PM10     365
NO2        0
SO2        0
CO         0
O3       365
dtype: int64

Dominant pollutant:
dominant_pollutant
PM10     73.7
PM2.5    19.5
O3        6.8
Name: proportion, dtype: float64

Saved. Master shape: (15283, 21)


In [21]:
import os, sys, importlib
import numpy as np
import pandas as pd

while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.append('src')
import aqi_calculator
importlib.reload(aqi_calculator)
from aqi_calculator import aqi_and_dominant

BAD = [5628, 6267807]
MIN_DAYS = 300

rec = pd.read_csv('data/raw/openaq/openaq_recent.csv', parse_dates=['date'])
loc = pd.read_csv('data/processed/openaq_locations_filtered.csv')[['location_id', 'latitude', 'longitude']]
rec = rec[~rec['location_id'].isin(BAD) & (rec['date'] < '2026-08-01')]

pm_map = {'pm25': 'PM2.5', 'pm10': 'PM10', 'no2': 'NO2', 'so2': 'SO2', 'co': 'CO', 'o3': 'O3'}
bounds = {'PM2.5': (0, 1000), 'PM10': (0, 1000), 'NO2': (0, 500),
          'SO2': (0, 500), 'CO': (0, 50), 'O3': (0, 500)}

w = (rec.assign(parameter=rec['parameter'].map(pm_map))
        .pivot_table(index=['location_id', 'date'], columns='parameter', values='value', aggfunc='mean'))
for c, (lo, hi) in bounds.items():
    if c not in w:
        w[c] = np.nan
    w.loc[(w[c] < lo) | (w[c] > hi), c] = np.nan
w = w[list(bounds)]
w['AQI'], _ = aqi_and_dominant(w[list(bounds)], min_pollutants=1)
w = w.reset_index()

meta = rec.drop_duplicates('location_id')[['location_id', 'location_name', 'city']]
pollutants = (rec.assign(parameter=rec['parameter'].map(pm_map))
                 .groupby('location_id')['parameter']
                 .agg(lambda x: ','.join(sorted(set(x)))).rename('pollutants').reset_index())

stn = (w.dropna(subset=['AQI']).groupby('location_id')
         .agg(days=('date', 'nunique'), avg_aqi=('AQI', 'mean'),
              first_day=('date', 'min'), last_day=('date', 'max'))
         .reset_index()
         .merge(meta, on='location_id').merge(loc, on='location_id').merge(pollutants, on='location_id'))

print("Stations with any AQI, and how many pass the", MIN_DAYS, "day rule:")
print(pd.DataFrame({
    'with_AQI': stn.groupby('city').size(),
    'pass_rule': stn[stn['days'] >= MIN_DAYS].groupby('city').size()}).fillna(0).astype(int))

keep = stn[stn['days'] >= MIN_DAYS].copy()
print("\nAverage station AQI by city (stations passing the rule):")
print(keep.groupby('city')['avg_aqi'].agg(['count', 'min', 'mean', 'max']).round(1))

print("\nHighest 5 and lowest 5 stations:")
cols = ['city', 'location_name', 'avg_aqi', 'days', 'pollutants']
print(keep.sort_values('avg_aqi', ascending=False)[cols].head(5).round(1).to_string(index=False))
print(keep.sort_values('avg_aqi')[cols].head(5).round(1).to_string(index=False))

keep.to_csv('data/processed/station_avg_aqi.csv', index=False)
print("\nSaved", len(keep), "stations to data/processed/station_avg_aqi.csv")

Stations with any AQI, and how many pass the 300 day rule:
           with_AQI  pass_rule
city                          
Bengaluru        10          4
Chennai           8          6
Delhi            35         26
Kolkata           6          5
Lucknow           5          5
Mumbai           28         22
Patna             1          1

Average station AQI by city (stations passing the rule):
           count    min   mean    max
city                                 
Bengaluru      4   46.8   62.6   73.6
Chennai        6   55.1   77.5   99.8
Delhi         26  126.8  189.8  244.7
Kolkata        5   66.1   89.5  108.9
Lucknow        5   71.4  118.2  142.8
Mumbai        22   51.1   84.0  113.6
Patna          1   82.3   82.3   82.3

Highest 5 and lowest 5 stations:
 city                  location_name  avg_aqi  days    pollutants
Delhi         Wazirpur, Delhi - DPCC    244.7   355          PM10
Delhi           Mundka, Delhi - DPCC    243.9   355    PM10,PM2.5
Delhi     Jahangirpuri, Delhi 

In [22]:
from sklearn.cluster import DBSCAN

try:
    import folium
    print("folium", folium.__version__)
except ImportError:
    print("folium MISSING")

stn = pd.read_csv('data/processed/station_avg_aqi.csv')

KM_PER_UNIT, AQI_PER_UNIT = 10.0, 25.0
x_km = stn['longitude'] * 111.32 * np.cos(np.radians(stn['latitude']))
y_km = stn['latitude'] * 110.57
X = np.column_stack([x_km / KM_PER_UNIT, y_km / KM_PER_UNIT, stn['avg_aqi'] / AQI_PER_UNIT])

for eps in [0.5, 1.0, 1.5]:
    stn[f'c_{eps}'] = DBSCAN(eps=eps, min_samples=3).fit_predict(X)

summary = []
for eps in [0.5, 1.0, 1.5]:
    lab = stn[f'c_{eps}']
    summary.append({'eps': eps, 'clusters': int(lab[lab >= 0].nunique()),
                    'noise_stations': int((lab == -1).sum()),
                    'noise_in_Delhi': int(((lab == -1) & (stn['city'] == 'Delhi')).sum()),
                    'noise_in_Mumbai': int(((lab == -1) & (stn['city'] == 'Mumbai')).sum())})
print(pd.DataFrame(summary).to_string(index=False))

for eps in [0.5, 1.0, 1.5]:
    lab = f'c_{eps}'
    cl = (stn[stn[lab] >= 0].groupby(lab)
            .agg(stations=('location_id', 'size'),
                 cities=('city', lambda s: ','.join(sorted(set(s)))),
                 mean_aqi=('avg_aqi', 'mean'), min_aqi=('avg_aqi', 'min'), max_aqi=('avg_aqi', 'max'))
            .round(1).sort_values('mean_aqi', ascending=False))
    print(f"\n--- eps = {eps} ---")
    print(cl.to_string())

folium 0.20.0
 eps  clusters  noise_stations  noise_in_Delhi  noise_in_Mumbai
 0.5         1              66              26               19
 1.0         9              13               3                3
 1.5         6               4               2                0

--- eps = 0.5 ---
       stations  cities  mean_aqi  min_aqi  max_aqi
c_0.5                                              
0             3  Mumbai      77.9     76.0     79.1

--- eps = 1.0 ---
       stations     cities  mean_aqi  min_aqi  max_aqi
c_1.0                                                 
5             4      Delhi     232.1    209.8    244.7
6             3      Delhi     197.5    189.6    209.7
0            16      Delhi     180.3    142.3    217.1
1             3    Lucknow     135.6    129.6    142.8
2            16     Mumbai      87.3     64.1    113.6
4             4    Kolkata      84.7     66.1     97.9
8             3     Mumbai      76.7     66.6     90.9
7             4    Chennai      71.3     

In [24]:
import folium
from folium import Element

PALETTE = ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd',
           '#8c564b', '#e377c2', '#17becf', '#bcbd22']
NOISE = '#9E9E9E'
HOTSPOT_MIN = 200

def build_map(df, col, title):
    cm = df[df[col] >= 0].groupby(col)['avg_aqi'].mean()
    order = list(cm.sort_values(ascending=False).index)
    color_of = {k: PALETTE[i % len(PALETTE)] for i, k in enumerate(order)}

    m = folium.Map(tiles='OpenStreetMap')
    for _, r in df.iterrows():
        k = r[col]
        is_noise = k < 0
        hot = (not is_noise) and cm[k] > HOTSPOT_MIN
        label = 'Noise (no cluster)' if is_noise else f'Cluster {k} (mean AQI {cm[k]:.0f})'
        popup = (f"<b>{r['location_name']}</b><br>{r['city']}<br>"
                 f"Average AQI: {r['avg_aqi']:.0f}<br>Days with AQI: {r['days']}<br>"
                 f"Pollutants: {r['pollutants']}<br>{label}")
        folium.CircleMarker(
            location=[r['latitude'], r['longitude']], radius=9,
            color='black' if hot else (NOISE if is_noise else color_of[k]),
            weight=3 if hot else 1,
            fill=True, fill_color=NOISE if is_noise else color_of[k], fill_opacity=0.85,
            popup=folium.Popup(popup, max_width=300),
            tooltip=f"{r['location_name']}: {r['avg_aqi']:.0f}").add_to(m)

    m.fit_bounds([[df['latitude'].min(), df['longitude'].min()],
                  [df['latitude'].max(), df['longitude'].max()]])

    rows = ''.join(
        f"<div><span style='background:{color_of[k]};display:inline-block;width:12px;height:12px;"
        f"border-radius:6px;margin-right:6px;'></span>Cluster {k}: "
        f"{', '.join(sorted(set(df[df[col] == k]['city'])))}, {int((df[col] == k).sum())} stations, "
        f"mean AQI {cm[k]:.0f}{' (hotspot)' if cm[k] > HOTSPOT_MIN else ''}</div>"
        for k in order)
    legend = (f"<div style='position:fixed;bottom:20px;left:20px;z-index:9999;background:white;"
              f"padding:10px;border:1px solid #888;font-size:12px;max-width:330px;'>"
              f"<b>{title}</b><br>{rows}"
              f"<div><span style='background:{NOISE};display:inline-block;width:12px;height:12px;"
              f"border-radius:6px;margin-right:6px;'></span>Noise: {int((df[col] < 0).sum())} stations</div>"
              f"<div style='margin-top:6px;color:#555;'>DBSCAN on location and average AQI, "
              f"Aug 2025 to Jul 2026. Black outline = cluster mean above {HOTSPOT_MIN}. "
              f"Station AQI depends on which pollutants a station measures.</div></div>")
    m.get_root().html.add_child(Element(legend))
    return m

os.makedirs('reports', exist_ok=True)
col = 'c_1.0'

build_map(stn[stn['city'] == 'Delhi'], col,
          'Delhi station clusters (eps = 1.0)').save('reports/hotspot_map_delhi.html')
build_map(stn, col,
          'Seven cities, station clusters (eps = 1.0)').save('reports/hotspot_map_india.html')

stn.to_csv('data/processed/station_clusters.csv', index=False)
print("Saved reports/hotspot_map_delhi.html, reports/hotspot_map_india.html, station_clusters.csv")

Saved reports/hotspot_map_delhi.html, reports/hotspot_map_india.html, station_clusters.csv
